# Project 1 ANSC 4040 Model Tuning

This is a notebook for Project 1 that will be used for expanding the same size and tuning the hyperparameters of original model

In [5]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [17]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [6]:
import pandas as pd
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Load Train and Validation Sets (100 cows)

In [7]:
train_path = "train_data.csv"
val_path = "val_data.csv"

# Find the training-set cow IDs using a bounded-memory scan.
animal_ids = set()
for id_chunk in pd.read_csv(
    train_path,
    usecols=["AnimalId"],
    dtype={"AnimalId": "string"},
    chunksize=100_000,
):
    animal_ids.update(id_chunk["AnimalId"].dropna().unique())

selected_animals = set(
    pd.Series(sorted(animal_ids)).sample(n=100, random_state=42).tolist()
)

# Read only model columns and rows for those 100 cows.
model_cols = [
    "AnimalId",
    "EventDate",
    "Avgmilkflow",
    "Flow30_60Session",
    "YieldFirst2Min_Session",
    "YieldSession",
    "DurationSession_sec",
    "milking",
]

def load_selected_cows(file_path):
    matching_chunks = []
    for chunk in pd.read_csv(
        file_path,
        usecols=model_cols,
        dtype={"AnimalId": "string"},
        chunksize=100_000,
    ):
        matching = chunk[chunk["AnimalId"].isin(selected_animals)]
        if not matching.empty:
            matching_chunks.append(matching)
    if not matching_chunks:
        raise ValueError(f"None of the selected cows were found in {file_path}.")
    return pd.concat(matching_chunks, ignore_index=True)

train = load_selected_cows(train_path)
val = load_selected_cows(val_path)

print(f"Selected cows: {sorted(selected_animals)}")
print(f"Training rows: {len(train):,}; validation rows: {len(val):,}")

# Split EventDate into numerical calendar features
for data in (train, val):
    event_date = pd.to_datetime(data["EventDate"])
    data["EventYear"] = event_date.dt.year
    data["EventMonth"] = event_date.dt.month
    data["EventDay"] = event_date.dt.day
    data["EventDayOfWeek"] = event_date.dt.dayofweek

Selected cows: ['-1.1716852323791606e+18', '-1.1833355166981594e+18', '-1.3332775441306954e+17', '-2.05144505093895e+18', '-2.099013255094703e+18', '-2.221464130034193e+18', '-2.25327613911556e+18', '-2.2990285093312376e+18', '-2.3422767415993856e+18', '-2.362711626198422e+18', '-2.4428249858373806e+18', '-2.49718279280027e+18', '-2.5715480822330004e+18', '-2.972840480631011e+17', '-3.1150506377499264e+18', '-3.2709915003359217e+18', '-3.455482512856752e+18', '-3.671253680800321e+18', '-3.7808412218609434e+18', '-3.788877104804816e+18', '-3.85672259016968e+18', '-4.1398554415718595e+18', '-4.225393355328677e+18', '-4.278792536301574e+18', '-4.4539072266346624e+18', '-4.482367362136449e+18', '-4.572979853174115e+18', '-4.725483000184784e+18', '-4.7479871501442355e+17', '-4.907405775221401e+18', '-5.274629919485563e+18', '-5.297614520013029e+17', '-5.398349683912395e+18', '-5.422953988785325e+18', '-5.451041495972229e+18', '-5.501116040247302e+18', '-5.556540390544463e+18', '-6.073988599

# Tuned Hyperparameters

In [8]:
# Target variable
target = "AnimalId"

# Define feature by including columns
feature_cols = ["Avgmilkflow", "EventYear", "EventMonth", "EventDay", "EventDayOfWeek", "Flow30_60Session", "YieldFirst2Min_Session", "YieldSession", "DurationSession_sec", "milking"]



X_train = train[feature_cols]
y_train = train[target]

X_val = val[feature_cols]
y_val = val[target]

rf_model = RandomForestClassifier(
        n_estimators=20,
        random_state=42,
        max_depth=5,
        min_samples_leaf=1,
        min_samples_split=2
    )

# Fit model
rf_model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",20
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",5
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: 

In [21]:
# Evaluate on validation rows that were not used to fit the model.
y_val_pred = rf_model.predict(X_val)
val_accuracy = accuracy_score(y_val, y_val_pred)

print(f"Validation accuracy: {val_accuracy:.2%}")
print("\nClassification report:")
print(classification_report(y_val, y_val_pred, zero_division=0))

Validation accuracy: 8.49%

Classification report:
                         precision    recall  f1-score   support

-1.1716852323791606e+18       0.00      0.00      0.00       228
-1.1833355166981594e+18       0.03      0.11      0.05        28
-1.3332775441306954e+17       0.00      0.00      0.00        59
  -2.05144505093895e+18       0.06      0.28      0.10       121
 -2.099013255094703e+18       0.05      0.08      0.06       315
 -2.221464130034193e+18       0.00      0.00      0.00       244
  -2.25327613911556e+18       0.05      0.24      0.08       147
-2.2990285093312376e+18       0.40      0.15      0.22       243
-2.3422767415993856e+18       0.00      0.00      0.00        45
 -2.362711626198422e+18       0.00      0.00      0.00       233
-2.4428249858373806e+18       0.00      0.00      0.00        99
  -2.49718279280027e+18       0.00      0.00      0.00       168
-2.5715480822330004e+18       0.00      0.00      0.00       209
 -2.972840480631011e+17       0.00    

# Hyperparameter tuning (now for 100 cows)

This small grid searches a realistic set of Random Forest settings to find the best combination without making the tuning process too slow or too broad.

In [22]:
from sklearn.model_selection import GridSearchCV

# Hyperparameter tuning with GridSearchCV
# This explores a small grid of RandomForest settings and selects the combination
# with the best cross-validation accuracy.

param_grid = {
    "n_estimators": [5, 50, 100],
    "max_depth": [None, 5, 10, 15],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 4, 8],
}

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    scoring="accuracy",
    cv=3,
    n_jobs=-1,
    verbose=1,
)

grid_search.fit(X_train, y_train)

print(f"Best cross-validation accuracy: {grid_search.best_score_:.2%}")
print(f"Best parameters: {grid_search.best_params_}")

best_model = grid_search.best_estimator_
y_val_pred_best = best_model.predict(X_val)
best_val_accuracy = accuracy_score(y_val, y_val_pred_best)

print(f"Validation accuracy with tuned model: {best_val_accuracy:.2%}")
print("\nClassification report for tuned model:")
print(classification_report(y_val, y_val_pred_best, zero_division=0))

results_df = pd.DataFrame(grid_search.cv_results_).sort_values("mean_test_score", ascending=False)
print("\nTop GridSearchCV results:")
print(results_df[["params", "mean_test_score", "std_test_score"]].head(10).to_string(index=False))

Fitting 3 folds for each of 108 candidates, totalling 324 fits


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/model_selection/_split.py:812: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Best cross-validation accuracy: 13.26%
Best parameters: {'max_depth': 15, 'min_samples_leaf': 4, 'min_samples_split': 2, 'n_estimators': 50}
Validation accuracy with tuned model: 11.63%

Classification report for tuned model:
                         precision    recall  f1-score   support

-1.1716852323791606e+18       0.29      0.36      0.32       228
-1.1833355166981594e+18       0.14      0.96      0.25        28
-1.3332775441306954e+17       0.00      0.00      0.00        59
  -2.05144505093895e+18       0.03      0.16      0.06       121
 -2.099013255094703e+18       0.01      0.00      0.00       315
 -2.221464130034193e+18       0.09      0.02      0.03       244
  -2.25327613911556e+18       0.06      0.24      0.09       147
-2.2990285093312376e+18       0.05      0.01      0.01       243
-2.3422767415993856e+18       1.00      0.29      0.45        45
 -2.362711626198422e+18       0.58      0.08      0.14       233
-2.4428249858373806e+18       0.11      0.33      0.17    

Comparison of 5 to 100 cow hyperparameters

5- {'max_depth': 5, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 20}
**100- {'max_depth': 15, 'min_samples_leaf': 4, 'min_samples_split': 2, 'n_estimators': 50}**

# Explore XGBoost as an alternative tree-based model

In [1]:
%pip install xgboost

  Using cached xgboost-3.4.1-py3-none-macosx_12_0_arm64.whl.metadata (2.0 kB)
  Using cached numpy-2.5.3-cp313-cp313-macosx_14_0_arm64.whl.metadata (6.6 kB)
  Using cached scipy-1.18.1-cp313-cp313-macosx_14_0_arm64.whl.metadata (62 kB)
Using cached xgboost-3.4.1-py3-none-macosx_12_0_arm64.whl (2.4 MB)
Using cached numpy-2.5.3-cp313-cp313-macosx_14_0_arm64.whl (5.4 MB)
Using cached scipy-1.18.1-cp313-cp313-macosx_14_0_arm64.whl (20.5 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [xgboost]m2/3 [xgboost]
Note: you may need to restart the kernel to use updated packages.


In [1]:
from xgboost import XGBClassifier

In [31]:
print(sklearn.__version__)

1.9.1


In [11]:

import numpy as np
from sklearn.metrics import f1_score, make_scorer
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

# Encode labels as consecutive integers for XGBoost multiclass classification.
animal_id_to_idx = {animal_id: i for i, animal_id in enumerate(sorted(selected_animals))}
y_train = pd.Series(train["AnimalId"].map(animal_id_to_idx), index=train.index, dtype="int64")
y_val = pd.Series(val["AnimalId"].map(animal_id_to_idx), index=val.index, dtype="int64")

# Keep singleton classes in every training fold; score only classes that can be split.
class_counts = y_train.value_counts()
cv_classes = class_counts[class_counts >= 2].index.sort_values().to_numpy()
tunable_indices = np.flatnonzero(y_train.isin(cv_classes).to_numpy())
rare_indices = np.flatnonzero(~y_train.isin(cv_classes).to_numpy())
if len(cv_classes) == 0:
    raise ValueError("At least one class needs two or more training rows for CV tuning.")

splitter = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)
cv_splits = [
    (np.concatenate((tunable_indices[train_part], rare_indices)), tunable_indices[val_part])
    for train_part, val_part in splitter.split(
        X_train.iloc[tunable_indices], y_train.iloc[tunable_indices]
    )
]

xgb_model = XGBClassifier(
    objective="multi:softprob",
    eval_metric="mlogloss",
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
)

# Twelve two-fold trials are a much smaller workload than an exhaustive grid.
param_distributions = {
    "n_estimators": [100, 200, 300],
    "max_depth": [3, 4, 5, 6],
    "learning_rate": [0.03, 0.05, 0.1],
    "min_child_weight": [1, 3, 5],
    "subsample": [0.7, 0.85, 1.0],
    "colsample_bytree": [0.7, 0.85, 1.0],
    "reg_lambda": [1, 5, 10],
}
search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_distributions,
    n_iter=12,
    scoring=make_scorer(
        f1_score, labels=cv_classes, average="macro", zero_division=0
    ),
    cv=cv_splits,
    refit=True,
    random_state=42,
    n_jobs=1,
    verbose=1,
    error_score="raise",
)
search.fit(X_train, y_train)

best_model = search.best_estimator_
y_val_pred = best_model.predict(X_val)
val_accuracy = accuracy_score(y_val, y_val_pred)
val_macro_f1 = f1_score(y_val, y_val_pred, average="macro", zero_division=0)
print(f"Best CV macro-F1: {search.best_score_:.2%}")
print(f"Best parameters: {search.best_params_}")
print(f"Validation accuracy: {val_accuracy:.2%}")
print(f"Validation macro-F1: {val_macro_f1:.2%}")
print(classification_report(y_val, y_val_pred, zero_division=0))

Validation accuracy: 9.37%

Classification report:
              precision    recall  f1-score   support

           0       0.26      0.19      0.22       228
           1       0.10      0.86      0.18        28
           2       0.00      0.00      0.00        59
           3       0.01      0.04      0.02       121
           4       0.01      0.01      0.01       315
           5       0.07      0.02      0.03       244
           6       0.04      0.14      0.06       147
           7       0.03      0.00      0.01       243
           8       1.00      0.27      0.42        45
           9       0.25      0.03      0.05       233
          10       0.00      0.01      0.01        99
          11       0.00      0.01      0.01       168
          12       0.01      0.00      0.01       209
          13       0.17      1.00      0.29        19
          14       0.02      0.04      0.02       204
          15       0.00      0.00      0.00        47
          16       0.10      0

In [12]:
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score

# Create the XGBoost classifier
xgb = XGBClassifier(
    objective="multi:softprob",
    eval_metric="mlogloss",
    random_state=42
)

# Hyperparameter search space
param_grid = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [3, 5, 7, 9],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "min_child_weight": [1, 3, 5, 7],
    "gamma": [0, 0.1, 0.3, 0.5],
    "reg_alpha": [0, 0.01, 0.1, 1],
    "reg_lambda": [1, 1.5, 2, 5]
}

# Randomized search
random_search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_grid,
    n_iter=50,
    scoring="accuracy",
    cv=5,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

# Run the optimization
random_search.fit(X_train, y_train)

# Best parameters
print("Best parameters:")
print(random_search.best_params_)

# Best cross-validation score
print("Best CV accuracy:")
print(random_search.best_score_)

# Best model
best_model = random_search.best_estimator_

# Evaluate on validation rows that were not used to fit the model.
y_val_pred = best_model.predict(X_val)
val_accuracy = accuracy_score(y_val, y_val_pred)

print(f"Validation accuracy: {val_accuracy:.2%}")
print("\nClassification report:")
print(classification_report(y_val, y_val_pred, zero_division=0))

Fitting 5 folds for each of 50 candidates, totalling 250 fits


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/model_selection/_split.py:812: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


[CV] END colsample_bytree=0.9, gamma=0, learning_rate=0.2, max_depth=9, min_child_weight=3, n_estimators=300, reg_alpha=1, reg_lambda=1, subsample=1.0; total time=   0.0s
[CV] END colsample_bytree=0.8, gamma=0.5, learning_rate=0.05, max_depth=9, min_child_weight=1, n_estimators=200, reg_alpha=0.1, reg_lambda=1.5, subsample=0.9; total time=   0.0s
[CV] END colsample_bytree=0.8, gamma=0.5, learning_rate=0.05, max_depth=9, min_child_weight=1, n_estimators=200, reg_alpha=0.1, reg_lambda=1.5, subsample=0.9; total time= 3.2min
[CV] END colsample_bytree=0.8, gamma=0.5, learning_rate=0.05, max_depth=9, min_child_weight=1, n_estimators=200, reg_alpha=0.1, reg_lambda=1.5, subsample=0.9; total time= 3.2min
[CV] END colsample_bytree=0.8, gamma=0.5, learning_rate=0.05, max_depth=9, min_child_weight=1, n_estimators=200, reg_alpha=0.1, reg_lambda=1.5, subsample=0.9; total time= 3.2min
[CV] END colsample_bytree=0.8, gamma=0.5, learning_rate=0.05, max_depth=9, min_child_weight=1, n_estimators=200, reg_

KeyboardInterrupt: 